# 07 — XGBoost Representation Comparison

## Objective

This notebook evaluates whether the findings observed with the Random Forest baseline are preserved when a different tree-based machine-learning algorithm, XGBoost, is used.

Three molecular representations are compared:

1. Morgan fingerprints — 2,048 features
2. RDKit descriptors — 205 features
3. Hybrid representation — 2,253 features

The same molecular dataset, target variable, train/test split, and representation alignment used in the Random Forest analysis are retained.

The analysis has two main objectives:

1. Compare the predictive performance of XGBoost across the three molecular representations.
2. Determine whether the relationship between chemical-space distance and prediction error remains representation-dependent when using XGBoost.

The primary comparison is therefore:

Molecular representation → XGBoost prediction → prediction error → chemical-space distance

This notebook is intended as a robustness analysis of the Random Forest findings.

# Import Libraries

In [1]:
import numpy as np
import pandas as pd

from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from scipy.stats import spearmanr

print("Libraries loaded successfully.")

Libraries loaded successfully.


# Verify the three representations

In [3]:
print("Available variables:")
print([v for v in dir() if v.startswith(("X_", "y", "train", "test"))])

Available variables:
[]


In [4]:
import os

print("Current directory:")
print(os.getcwd())

print("\nFiles:")
print(os.listdir())

Current directory:
C:\Users\kiran\Cheminformatics_course\Paper

Files:
['.ipynb_checkpoints', '01_DATA', '02_Caco2_Papp_Curated.csv', '02_Data_Set_Audit.ipynb', '03_EDA.ipynb', '04_TDA.ipynb', '05_Chemical_Space_Representation.ipynb', '06_Baseline Prediction Model.ipynb', '07_XGBoost_Representation_Comparison.ipynb', 'Caco2_chemical_space_master.pkl', 'Caco2_chemical_space_primary_results.pkl', 'caco2_morgan_chemical_space.pkl', 'Caco2_Papp_20449_single_fragment_dataset.csv', 'chembl_hia_raw_from_api.csv', 'Hybrid_chemical_space.pkl', 'RDKit_descriptor_chemical_space.pkl']


# Load the saved chemical-space data

In [6]:
import pickle
import numpy as np
import pandas as pd

# Load master chemical-space dataset
with open("Caco2_chemical_space_master.pkl", "rb") as f:
    master_data = pickle.load(f)

print(type(master_data))

if isinstance(master_data, dict):
    print(master_data.keys())

<class 'pandas.core.frame.DataFrame'>


In [7]:
# Inspect the master dataset

print("Shape:", master_data.shape)

print("\nColumns:")
print(master_data.columns.tolist())

print("\nFirst 3 rows:")
display(master_data.head(3))

Shape: (20449, 5)

Columns:
['canonical_smiles', 'log10_median_papp', 'Morgan_distance', 'RDKit_distance', 'Hybrid_distance']

First 3 rows:


,canonical_smiles,log10_median_papp,Morgan_distance,RDKit_distance,Hybrid_distance
0,Brc1ccc(-c2nc(-c3ccc4nc[nH]c4c3)no2)cc1,1.047664,0.200000,2.864551,0.539836
1,Brc1ccc(-c2nc3ccc(Br)cn3n2)cc1,0.542825,0.575000,6.721927,1.051168
2,Brc1ccc(-c2nnc(N3CCN(c4ccccn4)CC3)o2)cc1,0.770852,0.276596,6.929545,1.041043


# Load and inspect representations

In [8]:
import pickle

# Load Morgan chemical-space data
with open("caco2_morgan_chemical_space.pkl", "rb") as f:
    morgan_data = pickle.load(f)

# Load RDKit descriptor data
with open("RDKit_descriptor_chemical_space.pkl", "rb") as f:
    rdkit_data = pickle.load(f)

# Load Hybrid representation data
with open("Hybrid_chemical_space.pkl", "rb") as f:
    hybrid_data = pickle.load(f)

print("Morgan object:", type(morgan_data))
print("RDKit object :", type(rdkit_data))
print("Hybrid object:", type(hybrid_data))

print("\nMorgan keys:")
print(morgan_data.keys() if isinstance(morgan_data, dict) else "Not a dictionary")

print("\nRDKit keys:")
print(rdkit_data.keys() if isinstance(rdkit_data, dict) else "Not a dictionary")

print("\nHybrid keys:")
print(hybrid_data.keys() if isinstance(hybrid_data, dict) else "Not a dictionary")

Morgan object: <class 'dict'>
RDKit object : <class 'dict'>
Hybrid object: <class 'dict'>

Morgan keys:
dict_keys(['df_cs', 'X_morgan', 'nearest_indices', 'nearest_similarities', 'nearest_distances'])

RDKit keys:
dict_keys(['molecules', 'descriptors', 'descriptors_scaled', 'descriptor_names', 'nearest_neighbor_distance'])

Hybrid keys:
dict_keys(['molecules', 'morgan_normalized', 'rdkit_normalized', 'hybrid_representation', 'nearest_neighbor_distance', 'nearest_neighbor_index'])


# Restore the three representations

In [9]:
# Restore the three representations

# Restore representations

X_morgan = morgan_data["X_morgan"]

X_rdkit_scaled = rdkit_data["descriptors_scaled"]

X_hybrid = hybrid_data["hybrid_representation"]

# Restore target
y = master_data["log10_median_papp"].values

print("Morgan :", X_morgan.shape)
print("RDKit  :", X_rdkit_scaled.shape)
print("Hybrid :", X_hybrid.shape)
print("Target :", y.shape)

Morgan : (20449, 2048)
RDKit  : (20449, 205)
Hybrid : (20449, 2253)
Target : (20449,)


# Recreate the exact train/test split

In [10]:
from sklearn.model_selection import train_test_split

# Recreate the exact train/test split
indices = np.arange(len(y))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=42
)

print("Train indices:", len(train_idx))
print("Test indices :", len(test_idx))

print("\nFirst 10 test indices:")
print(test_idx[:10])

Train indices: 16359
Test indices : 4090

First 10 test indices:
[ 2300 10519  4148 15808  9161 14967 17622  4186 10502 15242]


# Create train/test matrices

In [11]:
# Create train/test sets using the exact same indices

X_morgan_train = X_morgan[train_idx]
X_morgan_test  = X_morgan[test_idx]

X_rdkit_train = X_rdkit_scaled[train_idx]
X_rdkit_test  = X_rdkit_scaled[test_idx]

X_hybrid_train = X_hybrid[train_idx]
X_hybrid_test  = X_hybrid[test_idx]

y_train = y[train_idx]
y_test  = y[test_idx]

print("=== Training Shapes ===")
print("Morgan :", X_morgan_train.shape)
print("RDKit  :", X_rdkit_train.shape)
print("Hybrid :", X_hybrid_train.shape)
print("y_train:", y_train.shape)

print("\n=== Test Shapes ===")
print("Morgan :", X_morgan_test.shape)
print("RDKit  :", X_rdkit_test.shape)
print("Hybrid :", X_hybrid_test.shape)
print("y_test :", y_test.shape)

=== Training Shapes ===
Morgan : (16359, 2048)
RDKit  : (16359, 205)
Hybrid : (16359, 2253)
y_train: (16359,)

=== Test Shapes ===
Morgan : (4090, 2048)
RDKit  : (4090, 205)
Hybrid : (4090, 2253)
y_test : (4090,)


# Define the controlled XGBoost model

In [12]:
from xgboost import XGBRegressor

xgb_params = {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1
}

print("XGBoost baseline parameters:")
for key, value in xgb_params.items():
    print(f"{key}: {value}")

XGBoost baseline parameters:
n_estimators: 300
learning_rate: 0.05
max_depth: 6
subsample: 0.8
colsample_bytree: 0.8
objective: reg:squarederror
random_state: 42
n_jobs: -1


# create the three models:

In [13]:
xgb_morgan = XGBRegressor(**xgb_params)
xgb_rdkit = XGBRegressor(**xgb_params)
xgb_hybrid = XGBRegressor(**xgb_params)

print("\nThree XGBoost models created successfully.")


Three XGBoost models created successfully.


# Train XGBoost on Morgan

In [14]:
print("Training XGBoost — Morgan representation...")

xgb_morgan.fit(
    X_morgan_train,
    y_train,
    verbose=False
)

print("Morgan XGBoost training completed.")

Training XGBoost — Morgan representation...
Morgan XGBoost training completed.


# Evaluation

In [15]:
# Morgan predictions
morgan_xgb_pred = xgb_morgan.predict(X_morgan_test)

# Performance metrics
morgan_xgb_mae = mean_absolute_error(y_test, morgan_xgb_pred)
morgan_xgb_rmse = np.sqrt(mean_squared_error(y_test, morgan_xgb_pred))
morgan_xgb_r2 = r2_score(y_test, morgan_xgb_pred)

print("=== XGBoost — Morgan ===")
print(f"MAE  : {morgan_xgb_mae:.4f}")
print(f"RMSE : {morgan_xgb_rmse:.4f}")
print(f"R²   : {morgan_xgb_r2:.4f}")

=== XGBoost — Morgan ===
MAE  : 0.4546
RMSE : 0.6054
R²   : 0.4600


# Train XGBoost on RDKit

In [17]:
print("Training XGBoost — RDKit representation...")

xgb_rdkit.fit(
    X_rdkit_train,
    y_train,
    verbose=False
)

print("RDKit XGBoost training completed.")

# Predictions
rdkit_xgb_pred = xgb_rdkit.predict(X_rdkit_test)

# Metrics
rdkit_xgb_mae = mean_absolute_error(y_test, rdkit_xgb_pred)
rdkit_xgb_rmse = np.sqrt(mean_squared_error(y_test, rdkit_xgb_pred))
rdkit_xgb_r2 = r2_score(y_test, rdkit_xgb_pred)

print("\n=== XGBoost — RDKit ===")
print(f"MAE  : {rdkit_xgb_mae:.4f}")
print(f"RMSE : {rdkit_xgb_rmse:.4f}")
print(f"R²   : {rdkit_xgb_r2:.4f}")

Training XGBoost — RDKit representation...
RDKit XGBoost training completed.

=== XGBoost — RDKit ===
MAE  : 0.4256
RMSE : 0.5775
R²   : 0.5086


# Train XGBoost on Hybrid

In [18]:
print("Training XGBoost — Hybrid representation...")

xgb_hybrid.fit(
    X_hybrid_train,
    y_train,
    verbose=False
)

print("Hybrid XGBoost training completed.")

# Predictions
hybrid_xgb_pred = xgb_hybrid.predict(X_hybrid_test)

# Metrics
hybrid_xgb_mae = mean_absolute_error(y_test, hybrid_xgb_pred)
hybrid_xgb_rmse = np.sqrt(mean_squared_error(y_test, hybrid_xgb_pred))
hybrid_xgb_r2 = r2_score(y_test, hybrid_xgb_pred)

print("\n=== XGBoost — Hybrid ===")
print(f"MAE  : {hybrid_xgb_mae:.4f}")
print(f"RMSE : {hybrid_xgb_rmse:.4f}")
print(f"R²   : {hybrid_xgb_r2:.4f}")

Training XGBoost — Hybrid representation...
Hybrid XGBoost training completed.

=== XGBoost — Hybrid ===
MAE  : 0.4377
RMSE : 0.5861
R²   : 0.4939


# XGBoost performance comparison

In [19]:
xgb_results = pd.DataFrame({
    "Representation": ["Morgan", "RDKit", "Hybrid"],
    "MAE": [
        morgan_xgb_mae,
        rdkit_xgb_mae,
        hybrid_xgb_mae
    ],
    "RMSE": [
        morgan_xgb_rmse,
        rdkit_xgb_rmse,
        hybrid_xgb_rmse
    ],
    "R2": [
        morgan_xgb_r2,
        rdkit_xgb_r2,
        hybrid_xgb_r2
    ]
})

display(xgb_results.round(4))

,Representation,MAE,RMSE,R2
0,Morgan,0.4546,0.6054,0.4600
1,RDKit,0.4256,0.5775,0.5086
2,Hybrid,0.4377,0.5861,0.4939


# Calculate XGBoost prediction errors

We already have the three prediction arrays:

- morgan_xgb_pred
- rdkit_xgb_pred
- hybrid_xgb_pred

In [20]:
# Calculate absolute prediction errors

morgan_xgb_error = np.abs(y_test - morgan_xgb_pred)
rdkit_xgb_error = np.abs(y_test - rdkit_xgb_pred)
hybrid_xgb_error = np.abs(y_test - hybrid_xgb_pred)

print("=== XGBoost Prediction Error Summary ===")

print("\nMorgan")
print("Mean absolute error   :", np.mean(morgan_xgb_error))
print("Median absolute error :", np.median(morgan_xgb_error))
print("Maximum absolute error:", np.max(morgan_xgb_error))

print("\nRDKit")
print("Mean absolute error   :", np.mean(rdkit_xgb_error))
print("Median absolute error :", np.median(rdkit_xgb_error))
print("Maximum absolute error:", np.max(rdkit_xgb_error))

print("\nHybrid")
print("Mean absolute error   :", np.mean(hybrid_xgb_error))
print("Median absolute error :", np.median(hybrid_xgb_error))
print("Maximum absolute error:", np.max(hybrid_xgb_error))

=== XGBoost Prediction Error Summary ===

Morgan
Mean absolute error   : 0.45460174279508275
Median absolute error : 0.3556588564249565
Maximum absolute error: 3.8768694400787354

RDKit
Mean absolute error   : 0.4256371985852173
Median absolute error : 0.32473857284300733
Maximum absolute error: 4.430104970932007

Hybrid
Mean absolute error   : 0.4377186528451098
Median absolute error : 0.3367169205325944
Maximum absolute error: 4.528498411178589


# Build the XGBoost analysis table

In [21]:
xgb_prediction_results = master_data.iloc[test_idx].copy()

xgb_prediction_results["y_true"] = y_test

xgb_prediction_results["Morgan_pred"] = morgan_xgb_pred
xgb_prediction_results["RDKit_pred"] = rdkit_xgb_pred
xgb_prediction_results["Hybrid_pred"] = hybrid_xgb_pred

xgb_prediction_results["Morgan_error"] = morgan_xgb_error
xgb_prediction_results["RDKit_error"] = rdkit_xgb_error
xgb_prediction_results["Hybrid_error"] = hybrid_xgb_error

print("Shape:", xgb_prediction_results.shape)

display(xgb_prediction_results.head())

Shape: (4090, 12)


,canonical_smiles,log10_median_papp,Morgan_distance,RDKit_distance,Hybrid_distance,y_true,Morgan_pred,RDKit_pred,Hybrid_pred,Morgan_error,RDKit_error,Hybrid_error
2300,CC(C)[C@H](NC(=O)CNC(=O)[C@@H](N)CCCNC(=N)N)C(...,-0.585027,0.213740,10.268270,0.506302,-0.585027,-0.644386,-1.038036,-0.951637,0.059360,0.453009,0.366610
10519,COc1nc(-c2cn([C@@H]3[C@@H](O)[C@@H](Sc4cc(Cl)c...,1.562293,0.205882,4.303679,0.555491,1.562293,1.187871,1.342518,1.086821,0.374422,0.219775,0.475471
4148,CCCn1c(=O)[nH]c(=O)c2c1nc(Cc1nc3cc(O)ccc3[nH]1...,1.547775,0.317460,3.466691,0.730767,1.547775,0.669011,0.367035,0.473818,0.878764,1.180739,1.073957
15808,N#Cc1c(-c2ccccc2)no[n+]1[O-],0.000000,0.285714,3.274167,0.631059,0.000000,1.270579,1.031833,0.892929,1.270579,1.031833,0.892929
9161,COc1cc2nc(C(=O)NCCC3CCN(Cc4ccccc4)CC3)cc(OC)c2...,1.113943,0.343284,4.982076,0.862701,1.113943,0.955396,0.842062,1.086569,0.158547,0.271882,0.027374


# Distance → prediction-error relationship

We'll use the matching representation for each analysis:

- Morgan distance → Morgan XGBoost error
- RDKit distance → RDKit XGBoost error
- Hybrid distance → Hybrid XGBoost error

In [23]:
# Calculate Spearman distance-error correlations

morgan_rho, morgan_p = spearmanr(
    xgb_prediction_results["Morgan_distance"],
    xgb_prediction_results["Morgan_error"]
)

rdkit_rho, rdkit_p = spearmanr(
    xgb_prediction_results["RDKit_distance"],
    xgb_prediction_results["RDKit_error"]
)

hybrid_rho, hybrid_p = spearmanr(
    xgb_prediction_results["Hybrid_distance"],
    xgb_prediction_results["Hybrid_error"]
)

print("=== XGBoost Distance → Prediction Error ===")

print(f"Morgan : rho = {morgan_rho:.6f}, p = {morgan_p:.3e}")
print(f"RDKit  : rho = {rdkit_rho:.6f}, p = {rdkit_p:.3e}")
print(f"Hybrid : rho = {hybrid_rho:.6f}, p = {hybrid_p:.3e}")

=== XGBoost Distance → Prediction Error ===
Morgan : rho = 0.074994, p = 1.576e-06
RDKit  : rho = 0.168051, p = 2.734e-27
Hybrid : rho = 0.089336, p = 1.047e-08


# Q1 → Q5 error progression
This will tell us whether prediction error systematically increases from chemically close molecules to chemically distant molecules.

In [24]:
# Create distance quintiles and calculate mean prediction error

quintile_results = {}

for representation in ["Morgan", "RDKit", "Hybrid"]:
    
    distance_col = f"{representation}_distance"
    error_col = f"{representation}_error"
    
    temp = xgb_prediction_results[
        [distance_col, error_col]
    ].copy()
    
    temp["distance_quintile"] = pd.qcut(
        temp[distance_col],
        q=5,
        labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
    )
    
    summary = (
        temp.groupby("distance_quintile", observed=True)[error_col]
        .agg(["mean", "median", "count"])
        .reset_index()
    )
    
    quintile_results[representation] = summary
    
    print(f"\n=== {representation} ===")
    display(summary.round(4))


=== Morgan ===


,distance_quintile,mean,median,count
0,Q1,0.4221,0.3319,823
1,Q2,0.4264,0.3414,826
2,Q3,0.4381,0.3479,806
3,Q4,0.4673,0.3581,817
4,Q5,0.5192,0.4049,818



=== RDKit ===


,distance_quintile,mean,median,count
0,Q1,0.3528,0.2865,818
1,Q2,0.3517,0.2584,818
2,Q3,0.4167,0.3167,818
3,Q4,0.4712,0.3636,818
4,Q5,0.5358,0.4245,818



=== Hybrid ===


,distance_quintile,mean,median,count
0,Q1,0.4030,0.3219,818
1,Q2,0.3811,0.3023,818
2,Q3,0.4314,0.3322,818
3,Q4,0.4653,0.3585,818
4,Q5,0.5078,0.3853,818


# Q1 vs Q5 statistical test
Now let's formally test whether the difference between chemically close (Q1) and distant (Q5) molecules is statistically significant.

In [25]:
from scipy.stats import mannwhitneyu

q1q5_results = []

for representation in ["Morgan", "RDKit", "Hybrid"]:
    
    distance_col = f"{representation}_distance"
    error_col = f"{representation}_error"
    
    temp = xgb_prediction_results[
        [distance_col, error_col]
    ].copy()
    
    temp["distance_quintile"] = pd.qcut(
        temp[distance_col],
        q=5,
        labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
    )
    
    q1 = temp.loc[
        temp["distance_quintile"] == "Q1",
        error_col
    ]
    
    q5 = temp.loc[
        temp["distance_quintile"] == "Q5",
        error_col
    ]
    
    U, p = mannwhitneyu(
        q1,
        q5,
        alternative="two-sided"
    )
    
    rank_biserial = 2 * U / (len(q1) * len(q5)) - 1
    
    q1_mean = q1.mean()
    q5_mean = q5.mean()
    
    increase_pct = ((q5_mean - q1_mean) / q1_mean) * 100
    
    q1q5_results.append({
        "Representation": representation,
        "Q1_mean_error": q1_mean,
        "Q5_mean_error": q5_mean,
        "Increase_%": increase_pct,
        "U": U,
        "p_value": p,
        "rank_biserial": rank_biserial
    })

q1q5_xgb = pd.DataFrame(q1q5_results)

display(q1q5_xgb.round(6))

,Representation,Q1_mean_error,Q5_mean_error,Increase_%,U,p_value,rank_biserial
0,Morgan,0.422127,0.519228,23.002573,294939.0,0.000014,-0.123788
1,RDKit,0.352753,0.535818,51.896006,254486.0,0.000000,-0.239346
2,Hybrid,0.402958,0.507781,26.013286,297462.0,0.000103,-0.110891


# Holm correction

In [27]:
# Holm correction without statsmodels

p_values = q1q5_xgb["p_value"].values

# Sort p-values
order = np.argsort(p_values)
sorted_p = p_values[order]

# Holm adjusted p-values
holm_sorted = np.maximum.accumulate(
    (len(sorted_p) - np.arange(len(sorted_p))) * sorted_p
)

# Restore original order
p_holm = np.empty_like(holm_sorted)
p_holm[order] = holm_sorted

# Add results
q1q5_xgb["p_holm"] = p_holm
q1q5_xgb["significant_after_holm"] = q1q5_xgb["p_holm"] < 0.05

display(
    q1q5_xgb[
        [
            "Representation",
            "Q1_mean_error",
            "Q5_mean_error",
            "Increase_%",
            "p_value",
            "p_holm",
            "rank_biserial",
            "significant_after_holm"
        ]
    ].round(6)
)

,Representation,Q1_mean_error,Q5_mean_error,Increase_%,p_value,p_holm,rank_biserial,significant_after_holm
0,Morgan,0.422127,0.519228,23.002573,0.000014,0.000028,-0.123788,True
1,RDKit,0.352753,0.535818,51.896006,0.000000,0.000000,-0.239346,True
2,Hybrid,0.402958,0.507781,26.013286,0.000103,0.000103,-0.110891,True


Prediction error increases with chemical-space distance, with the strongest distance-associated increase observed for the RDKit representation.

RDKit > Hybrid > Morgan

# RF vs XGBoost: distance–error robustness

In [28]:
rf_xgb_correlation = pd.DataFrame({
    "Representation": ["Morgan", "RDKit", "Hybrid"],
    "RF_Spearman_rho": [
        0.177002,
        0.250541,
        0.188698
    ],
    "XGBoost_Spearman_rho": [
        morgan_rho,
        rdkit_rho,
        hybrid_rho
    ]
})

rf_xgb_correlation["RF_Rank"] = (
    rf_xgb_correlation["RF_Spearman_rho"]
    .rank(ascending=False, method="min")
    .astype(int)
)

rf_xgb_correlation["XGBoost_Rank"] = (
    rf_xgb_correlation["XGBoost_Spearman_rho"]
    .rank(ascending=False, method="min")
    .astype(int)
)

display(rf_xgb_correlation.round(6))

,Representation,RF_Spearman_rho,XGBoost_Spearman_rho,RF_Rank,XGBoost_Rank
0,Morgan,0.177002,0.074994,3,3
1,RDKit,0.250541,0.168051,1,1
2,Hybrid,0.188698,0.089336,2,2


# Compare Q1→Q5 increases

In [29]:
rf_q1q5_increase = {
    "Morgan": 50.1,
    "RDKit": 80.6,
    "Hybrid": 57.7
}

rf_xgb_q1q5 = pd.DataFrame({
    "Representation": ["Morgan", "RDKit", "Hybrid"],
    "RF_Q1_Q5_increase_%": [
        rf_q1q5_increase["Morgan"],
        rf_q1q5_increase["RDKit"],
        rf_q1q5_increase["Hybrid"]
    ],
    "XGBoost_Q1_Q5_increase_%": q1q5_xgb["Increase_%"].values
})

display(rf_xgb_q1q5.round(2))

,Representation,RF_Q1_Q5_increase_%,XGBoost_Q1_Q5_increase_%
0,Morgan,50.1,23.00
1,RDKit,80.6,51.90
2,Hybrid,57.7,26.01


# LightGBM Robustness Analysis

## Objective

This notebook evaluates whether the representation-dependent relationship between
chemical-space distance and prediction error observed with Random Forest and XGBoost
is preserved when using LightGBM.

Three molecular representations are compared:

1. Morgan fingerprints — 2,048 features
2. RDKit descriptors — 205 features
3. Hybrid representation — 2,253 features

The same dataset, target variable, molecular ordering, and train/test split used in
the previous notebooks are retained.

The analysis consists of:

1. LightGBM predictive performance comparison
2. Prediction-error calculation
3. Chemical-space distance → prediction-error analysis
4. Distance quintile analysis
5. Q1 versus Q5 comparison
6. Comparison with Random Forest and XGBoost

The primary robustness question is whether the representation ranking observed in
the previous models is preserved with LightGBM.

In [31]:
import lightgbm as lgb

print("LightGBM version:", lgb.__version__)

LightGBM version: 4.6.0


# Define the controlled LightGBM baseline

In [32]:
from lightgbm import LGBMRegressor

lgb_params = {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 6,
    "num_leaves": 31,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1
}

print("LightGBM baseline parameters:")
for key, value in lgb_params.items():
    print(f"{key}: {value}")

LightGBM baseline parameters:
n_estimators: 300
learning_rate: 0.05
max_depth: 6
num_leaves: 31
subsample: 0.8
colsample_bytree: 0.8
random_state: 42
n_jobs: -1
verbosity: -1


In [33]:
lgb_morgan = LGBMRegressor(**lgb_params)
lgb_rdkit = LGBMRegressor(**lgb_params)
lgb_hybrid = LGBMRegressor(**lgb_params)

print("\nThree LightGBM models created successfully.")


Three LightGBM models created successfully.


# Train LightGBM on Morgan

In [34]:
print("Training LightGBM — Morgan representation...")

lgb_morgan.fit(
    X_morgan_train,
    y_train
)

print("Morgan LightGBM training completed.")

# Predictions
morgan_lgb_pred = lgb_morgan.predict(X_morgan_test)

# Metrics
morgan_lgb_mae = mean_absolute_error(y_test, morgan_lgb_pred)
morgan_lgb_rmse = np.sqrt(mean_squared_error(y_test, morgan_lgb_pred))
morgan_lgb_r2 = r2_score(y_test, morgan_lgb_pred)

print("\n=== LightGBM — Morgan ===")
print(f"MAE  : {morgan_lgb_mae:.4f}")
print(f"RMSE : {morgan_lgb_rmse:.4f}")
print(f"R²   : {morgan_lgb_r2:.4f}")

Training LightGBM — Morgan representation...
Morgan LightGBM training completed.

=== LightGBM — Morgan ===
MAE  : 0.4655
RMSE : 0.6240
R²   : 0.4262


C:\Users\kiran\anaconda3\envs\chemtox_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


# Train LightGBM on RDKit

In [35]:
print("Training LightGBM — RDKit representation...")

lgb_rdkit.fit(
    X_rdkit_train,
    y_train
)

print("RDKit LightGBM training completed.")

# Predictions
rdkit_lgb_pred = lgb_rdkit.predict(X_rdkit_test)

# Metrics
rdkit_lgb_mae = mean_absolute_error(y_test, rdkit_lgb_pred)
rdkit_lgb_rmse = np.sqrt(mean_squared_error(y_test, rdkit_lgb_pred))
rdkit_lgb_r2 = r2_score(y_test, rdkit_lgb_pred)

print("\n=== LightGBM — RDKit ===")
print(f"MAE  : {rdkit_lgb_mae:.4f}")
print(f"RMSE : {rdkit_lgb_rmse:.4f}")
print(f"R²   : {rdkit_lgb_r2:.4f}")

Training LightGBM — RDKit representation...
RDKit LightGBM training completed.

=== LightGBM — RDKit ===
MAE  : 0.4421
RMSE : 0.5974
R²   : 0.4741


C:\Users\kiran\anaconda3\envs\chemtox_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


# Train LightGBM on Hybrid

In [36]:
print("Training LightGBM — Hybrid representation...")

lgb_hybrid.fit(
    X_hybrid_train,
    y_train
)

print("Hybrid LightGBM training completed.")

# Predictions
hybrid_lgb_pred = lgb_hybrid.predict(X_hybrid_test)

# Metrics
hybrid_lgb_mae = mean_absolute_error(y_test, hybrid_lgb_pred)
hybrid_lgb_rmse = np.sqrt(mean_squared_error(y_test, hybrid_lgb_pred))
hybrid_lgb_r2 = r2_score(y_test, hybrid_lgb_pred)

print("\n=== LightGBM — Hybrid ===")
print(f"MAE  : {hybrid_lgb_mae:.4f}")
print(f"RMSE : {hybrid_lgb_rmse:.4f}")
print(f"R²   : {hybrid_lgb_r2:.4f}")

Training LightGBM — Hybrid representation...
Hybrid LightGBM training completed.

=== LightGBM — Hybrid ===
MAE  : 0.4444
RMSE : 0.5972
R²   : 0.4745


C:\Users\kiran\anaconda3\envs\chemtox_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


# Calculate LightGBM prediction errors

In [37]:
# Calculate absolute prediction errors

morgan_lgb_error = np.abs(y_test - morgan_lgb_pred)
rdkit_lgb_error = np.abs(y_test - rdkit_lgb_pred)
hybrid_lgb_error = np.abs(y_test - hybrid_lgb_pred)

print("=== LightGBM Prediction Error Summary ===")

print("\nMorgan")
print("Mean absolute error   :", np.mean(morgan_lgb_error))
print("Median absolute error :", np.median(morgan_lgb_error))
print("Maximum absolute error:", np.max(morgan_lgb_error))

print("\nRDKit")
print("Mean absolute error   :", np.mean(rdkit_lgb_error))
print("Median absolute error :", np.median(rdkit_lgb_error))
print("Maximum absolute error:", np.max(rdkit_lgb_error))

print("\nHybrid")
print("Mean absolute error   :", np.mean(hybrid_lgb_error))
print("Median absolute error :", np.median(hybrid_lgb_error))
print("Maximum absolute error:", np.max(hybrid_lgb_error))

=== LightGBM Prediction Error Summary ===

Morgan
Mean absolute error   : 0.4654808511292542
Median absolute error : 0.3604425509363788
Maximum absolute error: 3.8800573994376597

RDKit
Mean absolute error   : 0.44213537660617674
Median absolute error : 0.33802536482265927
Maximum absolute error: 4.586753709365397

Hybrid
Mean absolute error   : 0.4444310466098045
Median absolute error : 0.3373724807667875
Maximum absolute error: 4.48351332312906


# create the analysis table:

In [38]:
# LightGBM prediction/error analysis table

lgb_prediction_results = master_data.iloc[test_idx].copy()

lgb_prediction_results["y_true"] = y_test

lgb_prediction_results["Morgan_pred"] = morgan_lgb_pred
lgb_prediction_results["RDKit_pred"] = rdkit_lgb_pred
lgb_prediction_results["Hybrid_pred"] = hybrid_lgb_pred

lgb_prediction_results["Morgan_error"] = morgan_lgb_error
lgb_prediction_results["RDKit_error"] = rdkit_lgb_error
lgb_prediction_results["Hybrid_error"] = hybrid_lgb_error

print("Shape:", lgb_prediction_results.shape)

display(lgb_prediction_results.head())

Shape: (4090, 12)


,canonical_smiles,log10_median_papp,Morgan_distance,RDKit_distance,Hybrid_distance,y_true,Morgan_pred,RDKit_pred,Hybrid_pred,Morgan_error,RDKit_error,Hybrid_error
2300,CC(C)[C@H](NC(=O)CNC(=O)[C@@H](N)CCCNC(=N)N)C(...,-0.585027,0.213740,10.268270,0.506302,-0.585027,-0.385923,-0.820211,-0.773768,0.199103,0.235185,0.188742
10519,COc1nc(-c2cn([C@@H]3[C@@H](O)[C@@H](Sc4cc(Cl)c...,1.562293,0.205882,4.303679,0.555491,1.562293,1.237747,1.333993,0.994739,0.324546,0.228300,0.567554
4148,CCCn1c(=O)[nH]c(=O)c2c1nc(Cc1nc3cc(O)ccc3[nH]1...,1.547775,0.317460,3.466691,0.730767,1.547775,0.706806,0.352817,0.452155,0.840968,1.194958,1.095620
15808,N#Cc1c(-c2ccccc2)no[n+]1[O-],0.000000,0.285714,3.274167,0.631059,0.000000,1.298784,0.802925,0.887786,1.298784,0.802925,0.887786
9161,COc1cc2nc(C(=O)NCCC3CCN(Cc4ccccc4)CC3)cc(OC)c2...,1.113943,0.343284,4.982076,0.862701,1.113943,0.943039,0.742785,0.914163,0.170904,0.371158,0.199780


# LightGBM distance → prediction-error correlation

In [39]:
# Calculate Spearman distance-error correlations for LightGBM

morgan_lgb_rho, morgan_lgb_p = spearmanr(
    lgb_prediction_results["Morgan_distance"],
    lgb_prediction_results["Morgan_error"]
)

rdkit_lgb_rho, rdkit_lgb_p = spearmanr(
    lgb_prediction_results["RDKit_distance"],
    lgb_prediction_results["RDKit_error"]
)

hybrid_lgb_rho, hybrid_lgb_p = spearmanr(
    lgb_prediction_results["Hybrid_distance"],
    lgb_prediction_results["Hybrid_error"]
)

print("=== LightGBM Distance → Prediction Error ===")

print(f"Morgan : rho = {morgan_lgb_rho:.6f}, p = {morgan_lgb_p:.3e}")
print(f"RDKit  : rho = {rdkit_lgb_rho:.6f}, p = {rdkit_lgb_p:.3e}")
print(f"Hybrid : rho = {hybrid_lgb_rho:.6f}, p = {hybrid_lgb_p:.3e}")

=== LightGBM Distance → Prediction Error ===
Morgan : rho = 0.075653, p = 1.274e-06
RDKit  : rho = 0.140555, p = 1.700e-19
Hybrid : rho = 0.085457, p = 4.410e-08


# LightGBM Q1–Q5 analysis

In [40]:
# LightGBM distance quintile analysis

lgb_quintile_results = {}

for representation in ["Morgan", "RDKit", "Hybrid"]:
    
    distance_col = f"{representation}_distance"
    error_col = f"{representation}_error"
    
    temp = lgb_prediction_results[
        [distance_col, error_col]
    ].copy()
    
    temp["distance_quintile"] = pd.qcut(
        temp[distance_col],
        q=5,
        labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
    )
    
    summary = (
        temp.groupby("distance_quintile", observed=True)[error_col]
        .agg(["mean", "median", "count"])
        .reset_index()
    )
    
    lgb_quintile_results[representation] = summary
    
    print(f"\n=== {representation} ===")
    display(summary.round(4))


=== Morgan ===


,distance_quintile,mean,median,count
0,Q1,0.4304,0.3395,823
1,Q2,0.4383,0.3556,826
2,Q3,0.4523,0.3561,806
3,Q4,0.4823,0.3575,817
4,Q5,0.5245,0.4112,818



=== RDKit ===


,distance_quintile,mean,median,count
0,Q1,0.3730,0.3003,818
1,Q2,0.3747,0.2819,818
2,Q3,0.4378,0.3299,818
3,Q4,0.4862,0.3714,818
4,Q5,0.5390,0.4126,818



=== Hybrid ===


,distance_quintile,mean,median,count
0,Q1,0.4123,0.3178,818
1,Q2,0.3891,0.3061,818
2,Q3,0.4379,0.3401,818
3,Q4,0.4755,0.3519,818
4,Q5,0.5073,0.3828,818


# Formal LightGBM Q1 vs Q5 test

In [41]:
from scipy.stats import mannwhitneyu

lgb_q1q5_results = []

for representation in ["Morgan", "RDKit", "Hybrid"]:
    
    distance_col = f"{representation}_distance"
    error_col = f"{representation}_error"
    
    temp = lgb_prediction_results[
        [distance_col, error_col]
    ].copy()
    
    temp["distance_quintile"] = pd.qcut(
        temp[distance_col],
        q=5,
        labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
    )
    
    q1 = temp.loc[
        temp["distance_quintile"] == "Q1",
        error_col
    ]
    
    q5 = temp.loc[
        temp["distance_quintile"] == "Q5",
        error_col
    ]
    
    U, p = mannwhitneyu(
        q1,
        q5,
        alternative="two-sided"
    )
    
    rank_biserial = 2 * U / (len(q1) * len(q5)) - 1
    
    q1_mean = q1.mean()
    q5_mean = q5.mean()
    
    increase_pct = ((q5_mean - q1_mean) / q1_mean) * 100
    
    lgb_q1q5_results.append({
        "Representation": representation,
        "Q1_mean_error": q1_mean,
        "Q5_mean_error": q5_mean,
        "Increase_%": increase_pct,
        "U": U,
        "p_value": p,
        "rank_biserial": rank_biserial
    })

lgb_q1q5 = pd.DataFrame(lgb_q1q5_results)

display(lgb_q1q5.round(6))

,Representation,Q1_mean_error,Q5_mean_error,Increase_%,U,p_value,rank_biserial
0,Morgan,0.430428,0.524456,21.845357,294537.0,0.000012,-0.124983
1,RDKit,0.372971,0.539030,44.523120,268041.0,0.000000,-0.198830
2,Hybrid,0.412264,0.507299,23.052085,300204.0,0.000323,-0.102695


# Apply Holm correction

In [42]:
# Holm correction for LightGBM Q1 vs Q5 comparisons

p_values = lgb_q1q5["p_value"].values

order = np.argsort(p_values)
sorted_p = p_values[order]

holm_sorted = np.maximum.accumulate(
    (len(sorted_p) - np.arange(len(sorted_p))) * sorted_p
)

p_holm = np.empty_like(holm_sorted)
p_holm[order] = holm_sorted

lgb_q1q5["p_holm"] = p_holm
lgb_q1q5["significant_after_holm"] = (
    lgb_q1q5["p_holm"] < 0.05
)

display(
    lgb_q1q5[
        [
            "Representation",
            "Q1_mean_error",
            "Q5_mean_error",
            "Increase_%",
            "p_value",
            "p_holm",
            "rank_biserial",
            "significant_after_holm"
        ]
    ].round(6)
)

,Representation,Q1_mean_error,Q5_mean_error,Increase_%,p_value,p_holm,rank_biserial,significant_after_holm
0,Morgan,0.430428,0.524456,21.845357,0.000012,0.000023,-0.124983,True
1,RDKit,0.372971,0.539030,44.523120,0.000000,0.000000,-0.198830,True
2,Hybrid,0.412264,0.507299,23.052085,0.000323,0.000323,-0.102695,True


# 07 — Model Robustness: XGBoost and LightGBM Representation Comparison

## Overview

This notebook extends the Random Forest baseline analysis by evaluating whether the
observed relationship between chemical-space distance and prediction error is
preserved when alternative tree-based machine-learning algorithms are used.

Two additional algorithms were evaluated:

1. XGBoost
2. LightGBM

For both algorithms, three molecular representations were compared:

- Morgan fingerprints — 2,048 features
- RDKit molecular descriptors — 205 features
- Hybrid representation — 2,253 features

The same curated dataset, molecular ordering, target variable, and train/test split
used in the Random Forest analysis were retained throughout.

The primary purpose of this notebook is therefore model robustness, rather than
model optimization.

The central question is:

> Does the relationship between chemical-space distance and prediction error,
> including its dependence on molecular representation, remain observable when a
> different machine-learning algorithm is used?
 ## Controlled Experimental Design

To isolate the effect of the learning algorithm, the experimental conditions were
kept consistent with the Random Forest analysis.

### Dataset

- Total molecules: 20,449
- Target: log10_median_papp
- Training molecules: 16,359
- Test molecules: 4,090
- Train/test split: 80/20
- Random state: 42

The same test molecules were evaluated for Morgan, RDKit, and Hybrid representations
within each model.

### Molecular representations

| Representation | Features |
|---|---:|
| Morgan fingerprints | 2,048 |
| RDKit descriptors | 205 |
| Hybrid | 2,253 |

The Hybrid representation combines the normalized Morgan fingerprint and normalized
RDKit descriptor spaces.

### Models

Three tree-based regression algorithms were considered:

1. Random Forest — baseline from Notebook 06
2. XGBoost — robustness model
3. LightGBM — robustness model

XGBoost and LightGBM were evaluated using fixed baseline hyperparameters rather than
extensive hyperparameter optimization.

This controlled design allows the representation-dependent distance-error relationship
to be examined across different learning algorithms without introducing additional
optimization as a major source of variation.
## XGBoost Baseline

XGBoost was introduced as the first robustness model.

The same three molecular representations were used with the same train/test split.

### XGBoost baseline configuration

- n_estimators = 300
- learning_rate = 0.05
- max_depth = 6
- subsample = 0.8
- colsample_bytree = 0.8
- objective = reg:squarederror
- random_state = 42
- n_jobs = -1

These parameters were intentionally kept fixed across all three representations.

The purpose of this analysis was not to identify the optimal XGBoost model, but to
determine whether the chemical-space distance/error relationship observed with Random
Forest is preserved under a different tree-based learning algorithm.
## XGBoost Predictive Performance

The controlled XGBoost models produced the following test-set performance:

| Representation | MAE | RMSE | R² |
|---|---:|---:|---:|
| Morgan | 0.4546 | 0.6054 | 0.4600 |
| RDKit | 0.4256 | 0.5775 | 0.5086 |
| Hybrid | 0.4377 | 0.5861 | 0.4939 |

Under this fixed XGBoost configuration, RDKit descriptors produced the lowest MAE
and RMSE and the highest R² among the three representations.

This differs from the Random Forest baseline, where Morgan produced the strongest
overall predictive performance.

This difference is important because it indicates that predictive performance is
influenced by the interaction between the molecular representation and the learning
algorithm.

However, predictive performance alone is not the primary endpoint of this notebook.
The main analysis concerns the relationship between chemical-space distance and
prediction error.
## XGBoost: Chemical-Space Distance and Prediction Error

For each representation, the chemical-space distance was paired with the prediction
error generated from the corresponding representation.

The following matched relationships were evaluated:

- Morgan distance → Morgan XGBoost error
- RDKit distance → RDKit XGBoost error
- Hybrid distance → Hybrid XGBoost error

Spearman rank correlation was used because the analysis focuses on the monotonic
relationship between chemical-space distance and prediction error and does not require
a linear relationship.

### Results

| Representation | Spearman ρ | p-value |
|---|---:|---:|
| Morgan | 0.0750 | 1.576 × 10⁻⁶ |
| RDKit | 0.1681 | 2.734 × 10⁻²⁷ |
| Hybrid | 0.0893 | 1.047 × 10⁻⁸ |

All three representations showed statistically significant positive associations
between chemical-space distance and prediction error.

The relative strength of the association was:

RDKit > Hybrid > Morgan

Thus, the representation-dependent ranking observed with Random Forest was preserved
with XGBoost, although the absolute correlation coefficients were lower.
## XGBoost: Distance Quintile Analysis

To provide an interpretable view of the distance-error relationship, test molecules
were divided into five equal-frequency distance groups (Q1–Q5).

Q1 represents molecules closest to the training chemical space, whereas Q5 represents
the most distant molecules according to the corresponding representation.

### Mean absolute prediction error

| Representation | Q1 | Q2 | Q3 | Q4 | Q5 |
|---|---:|---:|---:|---:|---:|
| Morgan | 0.4221 | 0.4264 | 0.4381 | 0.4673 | 0.5192 |
| RDKit | 0.3528 | 0.3517 | 0.4167 | 0.4712 | 0.5358 |
| Hybrid | 0.4030 | 0.3811 | 0.4314 | 0.4653 | 0.5078 |

Prediction error generally increased toward the more distant chemical-space
quintiles.

The increase from Q1 to Q5 was:

- Morgan: +23.0%
- RDKit: +51.9%
- Hybrid: +26.0%

RDKit therefore showed the largest increase in prediction error between chemically
close and chemically distant test molecules.
## XGBoost: Q1 versus Q5 Statistical Comparison

The Q1 and Q5 groups were compared using the Mann–Whitney U test.

Holm correction was applied across the three representation-specific comparisons.

All three Q1 versus Q5 comparisons remained statistically significant after Holm
correction.

### Interpretation

The statistically significant Q1–Q5 differences support the observation that
prediction error increases as molecules become more distant from the training
chemical space.

The largest effect was observed for the RDKit representation.

The rank-biserial effect sizes were relatively small-to-moderate, indicating that
statistical significance should not be interpreted as evidence of a large practical
effect for every molecule.

The main conclusion is therefore based on the consistent direction and representation
ranking rather than statistical significance alone.
# LightGBM Robustness Analysis

## Objective

LightGBM was introduced as a second independent robustness model to determine whether
the representation-dependent distance-error relationship observed with Random Forest
and XGBoost is preserved across another tree-based learning algorithm.

### LightGBM baseline configuration

- n_estimators = 300
- learning_rate = 0.05
- max_depth = 6
- num_leaves = 31
- subsample = 0.8
- colsample_bytree = 0.8
- random_state = 42
- n_jobs = -1

The same parameters were used for Morgan, RDKit, and Hybrid representations.
No hyperparameter optimization was performed.
## LightGBM Predictive Performance

The controlled LightGBM models produced the following test-set performance:

| Representation | MAE | RMSE | R² |
|---|---:|---:|---:|
| Morgan | 0.4655 | 0.6240 | 0.4262 |
| RDKit | 0.4421 | 0.5974 | 0.4741 |
| Hybrid | 0.4444 | 0.5972 | 0.4745 |

Under this fixed LightGBM configuration, Hybrid and RDKit produced very similar
predictive performance, with Hybrid having a marginally higher R² and RDKit having
a marginally lower MAE.

These results again demonstrate that the representation giving the strongest
predictive performance can vary somewhat with the learning algorithm.

The subsequent distance-error analysis therefore remains essential.
## LightGBM: Chemical-Space Distance and Prediction Error

The same representation-matched analysis used for Random Forest and XGBoost was
performed for LightGBM.

### Results

| Representation | Spearman ρ | p-value |
|---|---:|---:|
| Morgan | 0.0757 | 1.274 × 10⁻⁶ |
| RDKit | 0.1406 | 1.700 × 10⁻¹⁹ |
| Hybrid | 0.0855 | 4.410 × 10⁻⁸ |

All three representations showed statistically significant positive associations
between chemical-space distance and prediction error.

The relative strength was again:

RDKit > Hybrid > Morgan

Thus, the representation ranking observed with Random Forest and XGBoost was
independently reproduced with LightGBM.
## LightGBM: Distance Quintile Analysis

The mean prediction error across distance quintiles was:

| Representation | Q1 | Q2 | Q3 | Q4 | Q5 |
|---|---:|---:|---:|---:|---:|
| Morgan | 0.4304 | 0.4383 | 0.4523 | 0.4823 | 0.5245 |
| RDKit | 0.3730 | 0.3747 | 0.4378 | 0.4862 | 0.5390 |
| Hybrid | 0.4123 | 0.3891 | 0.4379 | 0.4755 | 0.5073 |

The corresponding Q1→Q5 increases were:

- Morgan: +21.85%
- RDKit: +44.52%
- Hybrid: +23.05%

Again, RDKit showed the largest increase in prediction error with increasing
chemical-space distance.
## LightGBM: Q1 versus Q5 Statistical Comparison

Mann–Whitney U tests were performed between the Q1 and Q5 groups for each
representation.

Holm correction was applied across the three comparisons.

All three representation-specific comparisons remained statistically significant
after Holm correction.

The largest rank-biserial effect magnitude was observed for RDKit.

Therefore, the LightGBM analysis independently supports the conclusion that molecules
located farther from the training chemical space tend to exhibit greater prediction
error, with the strongest representation-dependent effect observed for RDKit.
# Three-Model Robustness Analysis

## Comparison Across Random Forest, XGBoost, and LightGBM

The central distance-error relationship was evaluated using three different
tree-based regression algorithms.

### Spearman distance-error correlations

| Representation | Random Forest | XGBoost | LightGBM |
|---|---:|---:|---:|
| Morgan | 0.1770 | 0.0750 | 0.0757 |
| RDKit | 0.2505 | 0.1681 | 0.1406 |
| Hybrid | 0.1887 | 0.0893 | 0.0855 |

Although the absolute magnitude of the correlations varied between algorithms,
the representation ranking was identical in all three models:

RDKit > Hybrid > Morgan

### Q1 → Q5 increase in mean prediction error

| Representation | Random Forest | XGBoost | LightGBM |
|---|---:|---:|---:|
| Morgan | 50.1% | 23.0% | 21.85% |
| RDKit | 80.6% | 51.9% | 44.52% |
| Hybrid | 57.7% | 26.0% | 23.05% |

Again, RDKit consistently showed the largest increase in prediction error between
chemically close and chemically distant molecules.

## Main robustness finding

The absolute strength of the distance-error relationship depends on the learning
algorithm. However, the relative behavior of the molecular representations is
remarkably consistent.

Across Random Forest, XGBoost, and LightGBM:

> **RDKit descriptors consistently exhibited the strongest association between
> chemical-space distance and prediction error, followed by the Hybrid
> representation and then Morgan fingerprints.**

This provides evidence that the representation-dependent distance-error relationship
is not specific to a single tree-based learning algorithm.

Importantly, this analysis does not imply that RDKit is the universally best molecular
representation for prediction. Predictive performance and distance-dependent error
behavior are distinct properties and should be interpreted separately.
# Scientific Interpretation

The results demonstrate two related but distinct effects.

### 1. Predictive performance depends on representation and model

The representation producing the best predictive performance was not identical across
all algorithms.

For example:

- Random Forest performed best with Morgan fingerprints.
- XGBoost performed best with RDKit descriptors under the selected baseline
  configuration.
- LightGBM produced very similar RDKit and Hybrid performance.

Therefore, molecular representation should not be evaluated solely on one model or
one predictive metric.

### 2. Distance-dependent prediction error is representation-dependent

Despite differences in overall predictive performance, all three learning algorithms
showed the same representation ranking for the distance-error relationship:

RDKit > Hybrid > Morgan

This indicates that molecular representation influences not only how accurately a
model predicts permeability, but also how prediction error changes as molecules
move farther from the training chemical space.

This distinction is central to the research question of the study.

### Important interpretation

The results should not be interpreted as:

> "RDKit is the best representation."

Instead, the more appropriate interpretation is:

> "Different molecular representations encode chemical-space relationships that
> influence the dependence of prediction error on chemical-space distance."

The current robustness analysis provides support for this representation-dependent
interpretation across multiple tree-based learning algorithms.
# Limitations of the Current Analysis

Several limitations should be considered.

### 1. Random train/test split

The current analysis uses an 80/20 random split. Structurally related molecules
may therefore occur in both the training and test sets.

This can make the test set less challenging than a scaffold-based or other
chemically disjoint split.

### 2. Baseline model configurations

XGBoost and LightGBM were evaluated using fixed baseline hyperparameters rather than
extensive optimization.

Therefore, the predictive performance values should not be interpreted as the
maximum achievable performance of either algorithm.

### 3. Tree-based model family

Random Forest, XGBoost, and LightGBM are all tree-based approaches.

The present robustness analysis therefore establishes robustness across several
related tree-based learners, rather than across fundamentally different model
families.

### 4. Correlation is not causation

A positive association between chemical-space distance and prediction error does not
by itself establish a causal mechanism.

The analysis demonstrates an empirical relationship between chemical-space position
and prediction error.

### 5. Statistical significance versus practical magnitude

With 4,090 test molecules, relatively small correlations can produce very small
p-values.

Therefore, effect magnitude, consistency across representations, and robustness
across models are considered alongside statistical significance.
# Role of This Notebook in the Overall Research

This notebook provides the model robustness stage of the study.

The Random Forest analysis established the initial observation that prediction error
increases with chemical-space distance and that the strength of this relationship
depends on molecular representation.

XGBoost and LightGBM were then used to test whether this observation depended on the
choice of learning algorithm.

The key result was preserved across all three models:

RDKit > Hybrid > Morgan

for the strength of the distance-error relationship.

Therefore, the central observation is sufficiently robust that additional tree-based
models are not necessary at this stage.

The next analysis should focus on **generalization under more chemically challenging
splitting strategies**, particularly scaffold-based separation.

This will determine whether the observed distance-error behavior persists when
structurally related molecules are more strictly separated between training and
testing.
# Conclusion

This notebook evaluated the robustness of the representation-dependent
chemical-space distance → prediction-error relationship using XGBoost and LightGBM
in addition to the Random Forest baseline.

Across all three tree-based algorithms:

- Prediction error increased with chemical-space distance.
- The distance-error relationship was statistically significant for Morgan, RDKit,
  and Hybrid representations.
- RDKit consistently showed the strongest distance-error association.
- Hybrid consistently showed an intermediate relationship.
- Morgan showed the weakest distance-error association.
- The ranking RDKit > Hybrid > Morgan was preserved across all three algorithms.

The absolute correlation strengths varied between models, demonstrating that the
magnitude of the distance-error relationship is partly model-dependent.

However, the consistent representation ranking provides evidence that the observed
phenomenon is not simply an artifact of a particular tree-based learning algorithm.

The next stage of the research will evaluate whether these findings remain under
more chemically challenging generalization settings, particularly scaffold-based
splitting.